# Transcrição de áudio com `labdados`

<a href="https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/transcricao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Transcreve áudio (`.mp3`, `.wav`, `.m4a`, `.ogg`, `.flac`) usando Whisper. Modos:

1. **Nuvem** — Whisper Large V3 Turbo no escritório (rápido). Suporta diarização (separação de falantes) com `whisperx`.
2. **Local** — `faster-whisper` no runtime do Colab. Sem diarização. Para áudios curtos.

📘 Doc completa: <https://lab-dados.github.io/labdados-sdk>

## 1. Instalar

In [ ]:
# Modo nuvem (mínimo)
%pip install -q labdados

In [ ]:
# Para o modo local (faster-whisper). ffmpeg já vem no Colab.
%pip install -q 'labdados[transcricao]'

## 2. Subir um áudio

Use o widget do Colab para enviar seu próprio áudio (entrevista, audiência, podcast). Se preferir testar primeiro com um exemplo curto, descomente a célula seguinte.

In [ ]:
from google.colab import files  # noqa: E402

uploaded = files.upload()
audio_path = next(iter(uploaded))
print('Arquivo:', audio_path)

In [ ]:
# Alternativa: amostra de 12s em domínio público (Wikipedia Commons).
# !wget -q -O amostra.ogg https://upload.wikimedia.org/wikipedia/commons/c/c8/Example.ogg
# audio_path = 'amostra.ogg'

## 3. Modo nuvem — transcrição simples

Cole sua API key (peça uma em <https://labdados.fgv.br>).

In [ ]:
import os
from getpass import getpass

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

In [ ]:
import labdados

labdados.transcricao(
    arquivos=audio_path,
    api_key=os.environ['LABDADOS_API_KEY'],
    saida='transcricoes/',
    idioma='pt',
    formato='srt',         # 'srt' (legenda), 'txt' (texto corrido), 'vtt' (web)
)
!ls transcricoes/

## 4. Modo nuvem — diarização (vários falantes)

Para audiências, entrevistas em grupo etc., use `modelo='whisperx'` + `diarizacao=True`. Se você sabe quantas pessoas falam, passe em `num_falantes` para melhorar o resultado.

In [ ]:
labdados.transcricao(
    arquivos=audio_path,
    api_key=os.environ['LABDADOS_API_KEY'],
    saida='transcricoes_diar/',
    modelo='whisperx',
    diarizacao=True,
    num_falantes=0,        # 0 = detecta automaticamente
    idioma='pt',
    formato='srt',
)
!ls transcricoes_diar/

Cada trecho da legenda vem prefixado com `SPEAKER_00`, `SPEAKER_01`, ...

## 5. Modo local (sem API key)

Para áudios curtos (até alguns minutos), o `faster-whisper` no runtime do Colab dá conta. Use modelos `tiny`/`base`/`small` em CPU; `medium`/`large-v3` exigem GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
labdados.transcricao(
    arquivos=audio_path,
    local=True,
    modelo_local='small',   # tiny | base | small | medium | large-v3
    idioma='pt',
    formato='srt',
    saida='transcricoes_local/',
)
!cat transcricoes_local/*.srt | head -20

## Tabela rápida de modelos `faster-whisper`

| Modelo     | Tamanho | Qualidade | Velocidade em CPU |
|------------|--------:|-----------|-------------------|
| `tiny`     |    75MB | ★         | rápido            |
| `base`     |   145MB | ★★        | rápido            |
| `small`    |   480MB | ★★★       | médio             |
| `medium`   |   1.5GB | ★★★★      | lento             |
| `large-v3` |     3GB | ★★★★★     | muito lento (GPU) |

Doc completa dos parâmetros: <https://lab-dados.github.io/labdados-sdk/api/transcricao.html>